In [1]:
# 금융위원회 기업기본정보 수집
from __future__ import annotations

import json
import time
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import pandas as pd


SERVICE_KEY = "61beda3f35ceca2dbe71e4c4f950844ae9d02486cfcd9e02a1af19cf035e9ca9"
PAGE_NO = 4
NUM_OF_ROWS = 10_000
RESULT_TYPE = "json"
BASE_URL = "https://apis.data.go.kr/1160100/service/GetCorpBasicInfoService_V2"
OUTPUT_DIR = Path.cwd() / "금융위원회_기업기본정보_추가수집"

ENDPOINTS = [
    {"name": "기업개요", "path": "getCorpOutline_V2"},
    {"name": "계열회사", "path": "getAffiliate_V2"},
    {"name": "연결대상종속기업", "path": "getConsSubsComp_V2"},
]


def fetch_json(endpoint_path: str, retries: int = 3) -> dict:
    """공공데이터 API를 호출하고 JSON 응답을 반환합니다."""
    query = urlencode(
        {
            "serviceKey": SERVICE_KEY,
            "pageNo": PAGE_NO,
            "numOfRows": NUM_OF_ROWS,
            "resultType": RESULT_TYPE,
        }
    )
    request = Request(
        f"{BASE_URL}/{endpoint_path}?{query}",
        headers={"User-Agent": "mle-01-p2-team3/1.0"},
    )

    last_error = None
    for attempt in range(1, retries + 1):
        try:
            with urlopen(request, timeout=180) as response:
                return json.loads(response.read().decode("utf-8-sig"))
        except (HTTPError, URLError, TimeoutError, json.JSONDecodeError) as error:
            last_error = error
            if attempt < retries:
                time.sleep(attempt)

    raise RuntimeError(f"API 호출 실패: {last_error}") from last_error


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for endpoint in ENDPOINTS:
    endpoint_name = endpoint["name"]
    response = fetch_json(endpoint["path"])
    api_response = response.get("response", {})
    header = api_response.get("header", {})

    if header.get("resultCode") != "00":
        raise RuntimeError(
            f"{endpoint_name} API 오류: {header.get('resultMsg', '알 수 없는 오류')}"
        )

    raw_items = api_response.get("body", {}).get("items", {}).get("item", [])
    items = raw_items if isinstance(raw_items, list) else [raw_items] if raw_items else []

    output_file = OUTPUT_DIR / f"{endpoint_name}_페이지_{PAGE_NO}.csv"
    pd.DataFrame(items).to_csv(output_file, index=False, encoding="utf-8-sig")
    print(f"{endpoint_name}: {len(items):,}건 저장")
    print(f"파일: {output_file}")

기업개요: 10,000건 저장
파일: c:\Users\Playdata\Desktop\김동석\교과목-2\mle-01-p2-team3\notebooks\금융위원회_기업기본정보_추가수집\기업개요_페이지_4.csv
계열회사: 10,000건 저장
파일: c:\Users\Playdata\Desktop\김동석\교과목-2\mle-01-p2-team3\notebooks\금융위원회_기업기본정보_추가수집\계열회사_페이지_4.csv
연결대상종속기업: 224건 저장
파일: c:\Users\Playdata\Desktop\김동석\교과목-2\mle-01-p2-team3\notebooks\금융위원회_기업기본정보_추가수집\연결대상종속기업_페이지_4.csv
